[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/lecturas/S1_datos_mibici.ipynb)

# Los datos de MiBici, en corto

Unos 10 minutos. No se entrega. Es para que sepas qué hay en la tabla antes del Lab 1: de dónde salió, qué se quitó y qué se agregó.
Corre las dos celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dammg_utils import COLORES

du.estilo_dammg()
np.set_printoptions(precision=4, suppress=True)

## 1. De dónde vienen

MiBici, el sistema de bicis de Guadalajara, publica sus **viajes sueltos**: un renglón por viaje, con estación y hora de salida y de llegada.
Nosotros los juntamos por **estación y hora**: cuántas bicis salieron y cuántas llegaron en cada hora, de octubre de 2025 a septiembre de 2026.
Nos quedamos con las 40 estaciones con más salidas.

**¿Qué es una fila de esta tabla?**

*Tip: no es un viaje.*

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # para explorar usamos solo octubre a junio
print(df.shape)
df[["station_id", "hour", "hour_of_day", "is_weekend", "departures", "arrivals"]].head()

## 2. Las columnas que vamos a usar

| Columna | Qué es |
|---|---|
| `departures` | Cuántas bicis salieron de esa estación en esa hora. Es la $y$ del Lab 1 |
| `hour_of_day`, `is_weekend` | La hora (0 a 23) y si es sábado o domingo |
| `departures_lag_24h`, `departures_lag_168h` | Las salidas de la misma estación y hora, ayer y hace una semana |
| `particion` | Si la fila es de `train`, `val` (julio) o `test` (agosto y septiembre) |

Las demás (`arrivals`, `net_flow`, `future_net`, ...) las usamos en S2.

## 3. Un día típico

**¿A qué horas salen más bicis? ¿Se parece un sábado a un martes?**

In [ ]:
por_hora = train.groupby(["is_weekend", "hour_of_day"]).departures.mean().unstack(0)
por_hora.columns = ["entre semana", "fin de semana"]
por_hora.plot(marker="o", figsize=(8, 3.5))
plt.xlabel("hora del día")
plt.ylabel("salidas promedio por estación")
plt.xticks(range(0, 24, 2))
plt.show()

Entre semana hay dos picos, de ida y de regreso del trabajo. El fin de semana no. Por eso la hora y el tipo de día ayudan a predecir.

## 4. Las 40 estaciones

**¿Todas las estaciones se parecen?**

In [ ]:
total = train.groupby("station_id").departures.sum().sort_values()
total.plot.barh(figsize=(7, 7), width=0.8)
plt.xlabel("salidas en entrenamiento, octubre a junio")
plt.ylabel("estación")
plt.show()
print(f"La más usada tiene {total.max() / total.min():.1f} veces las salidas de la menos usada.")

Una recta que no sabe la estación las trata igual. Acuérdate de esta gráfica cuando la tabla por estación le gane a la recta.

## 5. Qué se quitó

- Los viajes sin fecha válida o sin estación de salida o de llegada.
- El identificador del usuario: no lo necesitamos, y así no hay datos personales.
- Las estaciones dadas de baja o sin capacidad registrada.
- Las primeras horas de cada estación: ahí todavía no hay "hace una semana" y los rezagos salen vacíos (`NaN`). Se quitaron con `dropna()`.

**¿Por qué la tabla empieza el 8 de octubre y no el 1?**

*Tip: el promedio de la semana necesita ocho días de historia: siete de semana más uno de anticipación.*

In [ ]:
print("primera hora:", df.hour.min())
print("última hora: ", df.hour.max())
print("filas por estación:", df.groupby("station_id").size().unique())

Las horas sin viajes **sí están**, con 0 salidas. Si las quitáramos, "hace 24 filas" ya no sería "hace 24 horas".

In [ ]:
print(f"Horas con 0 salidas: {(train.departures == 0).mean():.0%}")

## 6. Qué se agregó

Los rezagos no vienen de MiBici: los calculamos. Para cada estación se recorre su historia 24 o 168 horas hacia atrás con `shift`.

**¿El `shift(24)` de una estación da lo mismo que `departures_lag_24h`?**

In [ ]:
una = df[df.station_id == df.station_id.iloc[0]].sort_values("hour")
mi_rezago = una.departures.shift(24)
comparar = pd.DataFrame({"hour": una.hour, "departures": una.departures,
                         "shift(24)": mi_rezago, "departures_lag_24h": una.departures_lag_24h})
display(comparar.iloc[24:30])
print("¿Coinciden en toda la estación?", (mi_rezago.iloc[24:] == una.departures_lag_24h.iloc[24:]).all())

Las primeras 24 filas salen vacías con nuestro `shift`, porque la tabla ya empieza recortada; la columna del archivo se calculó con la historia completa.
`departures_mean_168h` es el promedio de los 7 días que terminan 24 horas antes. Todas las entradas usan solo el pasado: a las 8 a.m. de hoy no sabemos cuántas bicis van a salir a las 8 a.m. de hoy.

## 7. Train, julio y agosto-septiembre

**¿Por qué partimos por fechas y no al azar?**

*Tip: piensa qué datos vas a tener el día que uses el modelo.*

In [ ]:
# Aquí sí vemos todo el año, pero solo las fechas: no decidimos nada con agosto y septiembre
dia = df.hour.dt.date
completos = df.groupby(dia).hour.nunique() == 24          # el primer y el último día vienen incompletos
por_dia = df[dia.map(completos)].groupby([dia, "particion"]).departures.sum().unstack()
por_dia[["train", "val", "test"]].plot(figsize=(9, 3.5), linewidth=1)
plt.xlabel("día")
plt.ylabel("salidas en el día (40 estaciones)")
plt.show()
df.particion.value_counts()

Con el modelo vamos a predecir días que todavía no pasan. Si revolviéramos al azar, el modelo vería el martes y el jueves para predecir el miércoles: el examen saldría fácil y engañoso.
Escogemos con `train`, comparamos con julio, y agosto y septiembre no se tocan hasta el final.

## En una línea

Cada fila es una estación en una hora; queremos predecir `departures` usando solo lo que ya pasó.